# Evo Aggregated Statistics to MongoDB

This notebook calculates aggregated statistics from Evo downhole objects using the MCP data analysis utilities (shared with the tools) and stores them in a MongoDB collection. 

**Objectives**: 
- Test a basic implementation of the Evo MCP utilities > mongo DB integration
- Experiment with the calculated statistics to determine what is useful
- Assess performance on querying those statistics over a number of objects

**Steps:**
- Connects to Evo platform via hijacked OAuth token 
- Calculates interval statistics (length-weighted mean, accumulation, etc.)
- Calculates per-hole statistics
- Stores results with timestamps in MongoDB for tracking
- Analyzes the performance with and without indexing

**Prerequisites:**
- MongoDB running locally 
- Evo MCP configured with valid credentials in `.env`
- `pymongo` installed

#### Setup

In [4]:
import sys
import pandas as pd
import json
from pathlib import Path
from uuid import UUID

# Add src directory to path for imports
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Add notebooks directory to path for supporting_scripts
notebooks_path = Path.cwd()
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

# MongoDB utilities
from supporting_scripts.mongo_utils import (
    connect_to_mongodb,
    estimate_doc_size,
    build_stats_summary,
    build_gap_summary,
    prepare_collection_documents,
    create_grade_stats_indexes,
    find_high_grade_objects,
    get_top_objects_by_grade,
    MONGO_DOC_LIMIT,
    SAFE_DOC_LIMIT,
)

# Benchmarking utilities
from supporting_scripts.stats_benchmark import (
    profile_query,
    get_explain_stats,
    run_index_benchmark,
    profile_high_grade_queries,
)

# Data loading utilities
from supporting_scripts.data_loading import download_all_interval_tables
from evo_mcp.utils.evo_data_utils import discover_objects, load_downhole_object

# Evo MCP utilities
from evo_mcp.context import evo_context, ensure_initialized
from evo_mcp.utils.data_analysis_utils import (
    calculate_interval_statistics,
    calculate_statistics_by_hole,
    analyze_gaps,
    calculate_multi_grade_statistics,
)

#### MongoDB onfiguration

Define the MongoDB connection settings and Evo workspace/object parameters.

In [15]:
# Load MongoDB connection parameters from config file

config_path = Path.cwd() / "mongo_config2.json"
if config_path.exists():
    with open(config_path, 'r') as f:
        mongo_config = json.load(f)
    
    # Check if Atlas credentials are provided
    if "username" in mongo_config and "password" in mongo_config and "cluster_url" in mongo_config:
        protocol = mongo_config.get("protocol", "mongodb+srv")
        username = mongo_config["username"]
        password = mongo_config["password"]
        cluster_url = mongo_config["cluster_url"]
        MONGO_URI = f"{protocol}://{username}:{password}@{cluster_url}"
        print(f"Using MongoDB Atlas ({protocol}): {cluster_url.split('/')[0]}")
    else:
        # Fall back to local MongoDB
        protocol = mongo_config.get("protocol", "mongodb")
        host = mongo_config.get("host", "localhost")
        port = mongo_config.get("port", 27017)
        MONGO_URI = f"{protocol}://{host}:{port}/"
        print(f"Using local MongoDB: {host}:{port}")
    
    MONGO_DB_NAME = mongo_config.get("database", "evo_statistics")
    MONGO_COLLECTION_NAME = mongo_config.get("collection", "interval_statistics")
else:
    print(f"Config file not found: {config_path}")
    MONGO_URI = "mongodb://localhost:27017/"
    MONGO_DB_NAME = "evo_statistics"
    MONGO_COLLECTION_NAME = "interval_statistics"

WORKSPACE_ID = "01c54ab3-0b97-4b36-8e72-686e65a906ed"  
# OBJECT_ID = "ea6ced6a-31d2-44a9-b8ae-1cee0b5ea42e"     # maia downhole intervals in _ultraenhance
OBJECT_ID = "0286ea01-1a2c-41a8-81b8-fca7df38feac"     # maia downhole collection in _ultraenhance

# Optional: specific version (leave empty for latest)
VERSION = ""

print(f"MongoDB: {MONGO_URI.split('@')[-1] if '@' in MONGO_URI else MONGO_URI}")
print(f"Database: {MONGO_DB_NAME}.{MONGO_COLLECTION_NAME}")
print(f"Evo Object: {OBJECT_ID}")

Config file not found: c:\Dev\evo-mcp-fork\notebooks\mongo_config2.json
MongoDB: mongodb://localhost:27017/
Database: evo_statistics.interval_statistics
Evo Object: 0286ea01-1a2c-41a8-81b8-fca7df38feac


## 3. Connect to MongoDB

Establish connection to MongoDB and create/access the target collection.

Make sure you've built the docker image for the server and that is running before executing this cell.The docker command to run the server is:

```bash
docker run -d -p 27017:27017 --name mongodb mongo:latest
```

In [16]:
mongo_client, mongo_db, stats_collection = connect_to_mongodb(MONGO_URI, MONGO_DB_NAME, MONGO_COLLECTION_NAME)

✓ Connected to MongoDB: evo_statistics.interval_statistics


## 4. Initialize Evo Connection

Initialize the Evo SDK context and authenticate via OAuth.

In [14]:
# Initialize Evo SDK connection (will trigger OAuth flow if needed)
await ensure_initialized()
print("✓ Evo SDK initialized and authenticated")

✓ Evo SDK initialized and authenticated


## 5. Load Object and Inspect Collections

Download the Evo object and inspect available collections/attributes.

In [18]:
# Discover all objects in the workspace, filtered to downhole types
all_objects = await discover_objects(
    WORKSPACE_ID,
    object_types=["downhole-collection", "downhole-intervals"],
)


print(f"Found {len(all_objects)} object(s) matching types ['downhole-collection', 'downhole-intervals']")
for o in all_objects:
    print(f"  {o['name']} ({o['schema_id']}) — {o['id']}")

print(f"\n{'='*70}\n")

# Load a specific object and inspect its collections
obj, obj_dict, object_name, object_type, collections_info = await load_downhole_object(WORKSPACE_ID, OBJECT_ID, VERSION)

print(f"Object: {object_name}")
print(f"Type: {object_type}")

total_attributes = 0
print(f"\nDiscovered {len(collections_info)} interval table(s):")
for coll in collections_info:
    attrs = coll.get('attributes', [])
    total_attributes += len(attrs)
    print(f"\n  {coll['name']} ({len(attrs)} attributes)")
    for attr in attrs:
        print(f"     - {attr['name']} ({attr['type']})")

print(f"\n{'='*50}")
print(f"Total: {len(collections_info)} tables × {total_attributes} attributes")
print(f"Will generate {len(collections_info)} MongoDB document(s) (one per interval table)")

Found 2 object(s) matching types ['downhole-collection', 'downhole-intervals']
  maia.json (downhole-collection) — 0286ea01-1a2c-41a8-81b8-fca7df38feac
  maia_geology_desurveyed.json (downhole-intervals) — ea6ced6a-31d2-44a9-b8ae-1cee0b5ea42e


Object: Maia Drillholes
Type: downhole-collection

Discovered 2 interval table(s):

  assay (1 attributes)
     - Au (continuous)

  geology (1 attributes)
     - Lithology (continuous)

Total: 2 tables × 2 attributes
Will generate 2 MongoDB document(s) (one per interval table)


## 6. Download All Interval Tables

Download interval data from every discovered collection. For each table, identify
which columns are numeric (suitable for statistics) vs categorical.

In [20]:
# Download all interval tables and classify their columns
import time

collection_data = await download_all_interval_tables(obj, object_type, collections_info)

for coll_name, coll_data in collection_data.items():
    print(f"\n{'='*50}")
    print(f"Downloaded: {coll_name}")
    df = coll_data["df"]
    numeric_cols = coll_data["numeric_cols"]
    categorical_cols = coll_data["categorical_cols"]
    elapsed = coll_data["elapsed"]
    print(f"  ✓ {len(df):,} intervals from {df['hole_id'].nunique():,} holes ({elapsed:.1f}s)")
    print(f"  Numeric attributes ({len(numeric_cols)}): {numeric_cols[:10]}{'...' if len(numeric_cols) > 10 else ''}")
    print(f"  Categorical attributes ({len(categorical_cols)}): {categorical_cols[:10]}{'...' if len(categorical_cols) > 10 else ''}")

print(f"\n{'='*50}")
print(f"Downloaded {len(collection_data)} interval table(s)")


Downloaded: assay
  ✓ 2,210 intervals from 14 holes (13.6s)
  Numeric attributes (1): ['Au']
  Categorical attributes (0): []

Downloaded: geology
  ✓ 93 intervals from 14 holes (10.8s)
  Numeric attributes (1): ['Lithology']
  Categorical attributes (0): []

Downloaded 2 interval table(s)


## 7. Calculate Statistics for All Attributes

For every numeric attribute on every interval table, compute:
- **Overall**: length-weighted mean, accumulation, min/max/std, count
- **Per-hole**: same metrics grouped by hole_id
- **Gaps**: missing interval analysis per collection

This can produce a large volume of data (e.g., Thalanga assays has ~100 attributes × 5,000 holes),
so we store **one MongoDB document per interval table** to keep documents manageable.

In [21]:
# Calculate statistics for every numeric attribute on every interval table
all_collection_stats = {}  # { collection_name: { "attributes": {...}, "gap_analysis": {...} } }

for coll_name, coll_data in collection_data.items():
    df = coll_data["df"]
    numeric_cols = coll_data["numeric_cols"]
    
    print(f"\n{'='*60}")
    print(f"📊 {coll_name}: {len(numeric_cols)} numeric attributes, {len(df):,} intervals")
    print(f"{'='*60}")
    
    t0 = time.perf_counter()
    attribute_stats = {}
    
    for grade_col in numeric_cols:
        # Overall statistics
        try:
            overall = calculate_interval_statistics(df, grade_col)
        except (ValueError, ZeroDivisionError):
            print(f"{grade_col} skipped (no valid numeric data)")
            continue
        
        # Per-hole statistics
        hole_stats_df = calculate_statistics_by_hole(df, grade_col)
        
        attribute_stats[grade_col] = {
            "overall": overall,
            "by_hole": hole_stats_df.to_dict(orient='records'),
            "hole_count": len(hole_stats_df),
        }
        
        lwm = overall.get('length_weighted_mean', 0)
        count = overall.get('count', 0)
        print(f"{grade_col}: LWM={lwm:.4f}, n={count}, holes={len(hole_stats_df)}")
    
    # Gap analysis (once per collection, not per attribute)
    gap_analysis = analyze_gaps(df)
    
    elapsed = time.perf_counter() - t0
    
    all_collection_stats[coll_name] = {
        "attributes": attribute_stats,
        "gap_analysis": gap_analysis,
    }
    
    print(f"\n  Gaps: {gap_analysis['total_gap_count']} total, {gap_analysis['holes_with_gaps']} holes affected")
    print(f"  Completed in {elapsed:.1f}s ({len(attribute_stats)} attributes analysed)")

print(f"\n{'='*60}")
print(f"Summary: {sum(len(v['attributes']) for v in all_collection_stats.values())} attribute statistics across {len(all_collection_stats)} table(s)")


📊 assay: 1 numeric attributes, 2,210 intervals
Au: LWM=0.4725, n=2210, holes=14

  Gaps: 0 total, 0 holes affected
  Completed in 0.1s (1 attributes analysed)

📊 geology: 1 numeric attributes, 93 intervals
Lithology: LWM=2.8053, n=93, holes=14

  Gaps: 0 total, 0 holes affected
  Completed in 0.0s (1 attributes analysed)

Summary: 2 attribute statistics across 2 table(s)


## 8. Prepare Documents for MongoDB

### Document Strategy

Each object may have multiple interval tables (e.g., `assay`, `geology`), and each table
may have many attributes. Storing everything in one document risks hitting MongoDB's **16 MB
document limit** — especially when per-hole statistics for 100+ attributes across 5,000+ holes
are included.

**Strategy: One document per interval table**, structured as:

```
{
  workspace_id, object_id, object_name, object_type,
  collection_name: "assay",
  stats_summary: [ {grade, lwm, accumulation, max, ...}, ... ],  // flat array for indexing
  grade_statistics: { "Au": {overall: {...}, by_hole: [...]}, ... },
  gap_analysis: {...},
  metadata: { version, timestamp, doc_size_bytes }
}
```

For extremely large tables (estimated doc > 14 MB), we split further into:
- **Summary document**: overall stats + stats_summary array (small, queryable)
- **Detail documents**: per-hole stats chunked by attribute batches

This keeps the summary docs fast to query while preserving full detail.

In [22]:
# Build all documents
all_documents = []

for coll_name, coll_stats in all_collection_stats.items():
    docs = prepare_collection_documents(
        workspace_id=WORKSPACE_ID,
        object_id=OBJECT_ID,
        object_name=object_name,
        object_type=object_type,
        collection_name=coll_name,
        attribute_stats=coll_stats["attributes"],
        gap_analysis=coll_stats["gap_analysis"],
    )
    all_documents.extend(docs)
    
    for doc in docs:
        size_mb = doc["metadata"].get("doc_size_bytes", 0) / 1024 / 1024
        n_attrs = len(doc.get("stats_summary", doc.get("attributes_in_chunk", [])))
        print(f"  📄 {coll_name} [{doc['doc_type']}]: {size_mb:.2f} MB, {n_attrs} attributes")

print(f"\nTotal documents to insert: {len(all_documents)}")
print(f"Total size: {sum(d['metadata'].get('doc_size_bytes', 0) for d in all_documents) / 1024 / 1024:.2f} MB")

  📄 assay [complete]: 0.00 MB, 1 attributes
  📄 geology [complete]: 0.00 MB, 1 attributes

Total documents to insert: 2
Total size: 0.01 MB


## 9. Insert Statistics into MongoDB

Write the statistics document to the MongoDB collection.

In [23]:
# Insert all documents into MongoDB
if all_documents:
    result = stats_collection.insert_many(all_documents)
    print(f"Inserted {len(result.inserted_ids)} document(s)")
    for i, doc_id in enumerate(result.inserted_ids):
        doc = all_documents[i]
        print(f"  {doc['collection_name']} [{doc['doc_type']}]: {doc_id}")
else:
    print("No documents to insert")

Inserted 2 document(s)
  assay [complete]: 69927639593201950417331e
  geology [complete]: 69927639593201950417331f


## 10. Create Grade Value Indexes

Create indexes to support queries like "find all high-grade objects" efficiently.

In [24]:
# Create the indexes
grade_indexes = create_grade_stats_indexes(stats_collection)

✓ Created 4 grade query indexes: ['grade_lwm', 'grade_accumulation', 'grade_max', 'workspace_grade_lwm']


## 10b. Index Performance Profiling

Compare query performance with and without indexes on the key query patterns.

In [25]:
# Run benchmark — documents are now present in the collection
doc_count = stats_collection.count_documents({})
print(f"Collection has {doc_count} documents")

if doc_count > 0:
    benchmark_results = run_index_benchmark(stats_collection, WORKSPACE_ID, OBJECT_ID)
else:
    print("No documents found — run the insert cell first")

Collection has 2 documents

Testing index: workspace_object_idx
Query: {'workspace_id': '01c54ab3-0b97-4b36-8e72-686e65a906ed', 'object_id': '0286ea01-1a2c-41a8-81b8-fca7df38feac'}

WITHOUT INDEX:
   Mean: 0.9725 ms (±0.2545)
   Docs examined: 2
   Index used: workspace_grade_lwm

WITH INDEX:
   Mean: 0.9346 ms (±0.1490)
   Docs examined: 2
   Keys examined: 2
   Index used: workspace_object_idx

   ⚡ Improvement: 3.9%

Testing index: object_name_idx
Query: {'object_name': {'$regex': '.*', '$options': 'i'}}

WITHOUT INDEX:
   Mean: 0.9657 ms (±0.1854)
   Docs examined: 2
   Index used: COLLSCAN

WITH INDEX:
   Mean: 0.9631 ms (±0.1855)
   Docs examined: 2
   Keys examined: 2
   Index used: object_name_idx

   ⚡ Improvement: 0.3%


## 10. Verify and Query MongoDB

Query the collection to verify the data was stored correctly and explore historical statistics.

In [26]:
# Collection overview
doc_count = stats_collection.count_documents({})
print(f"Total documents in collection: {doc_count}\n")

# Count by doc_type
for doc_type in ["complete", "summary", "detail"]:
    count = stats_collection.count_documents({"doc_type": doc_type})
    if count > 0:
        print(f"  {doc_type}: {count} document(s)")

# Show documents for this object
print(f"\nDocuments for object {OBJECT_ID}:")
cursor = stats_collection.find(
    {"object_id": OBJECT_ID},
    {"collection_name": 1, "doc_type": 1, "stats_summary": 1, "gap_analysis": 1, "timestamp": 1, "metadata.doc_size_bytes": 1}
).sort([("collection_name", 1), ("doc_type", 1)])

for doc in cursor:
    coll = doc.get("collection_name", "?")
    dtype = doc.get("doc_type", "?")
    size_kb = doc.get("metadata", {}).get("doc_size_bytes", 0) / 1024
    ts = doc.get("timestamp", "")
    
    if dtype in ("complete", "summary"):
        n_attrs = len(doc.get("stats_summary", []))
        gaps = doc.get("gap_analysis", {}).get("total_gap_count", 0)
        print(f"\n  📁 {coll} [{dtype}] — {n_attrs} attributes, {gaps} gaps, {size_kb:.0f} KB")
        
        # Show top 5 attributes by LWM
        summary = sorted(doc.get("stats_summary", []), key=lambda s: abs(s.get("lwm") or 0), reverse=True)
        for s in summary[:5]:
            print(f"     {s['grade']}: LWM={s.get('lwm', 'N/A')}, max={s.get('max', 'N/A')}, n={s.get('count', 'N/A')}")
        if len(summary) > 5:
            print(f"     ... and {len(summary) - 5} more")
    else:
        attrs = doc.get("attributes_in_chunk", [])
        print(f"\n  📁 {coll} [{dtype}] — {len(attrs)} attributes (detail chunk), {size_kb:.0f} KB")

Total documents in collection: 2

  complete: 2 document(s)

Documents for object 0286ea01-1a2c-41a8-81b8-fca7df38feac:

  📁 assay [complete] — 1 attributes, 0 gaps, 3 KB
     Au: LWM=0.4725068505663135, max=5.91, n=2210

  📁 geology [complete] — 1 attributes, 0 gaps, 3 KB
     Lithology: LWM=2.8052795031055897, max=5.0, n=93


## 11. Query High-Grade Objects

Example queries for finding objects by grade statistics.

In [28]:
# Example queries - uncomment to run:

# 1. Find all objects with Au length-weighted mean >= 1.0 g/t
high_au_objects = find_high_grade_objects(stats_collection, grade="Au", min_lwm=0.4)
print(f"Found {len(high_au_objects)} objects with Au LWM >= 1.0")

# 2. Find objects with Au peak values >= 10 g/t
# high_peak_objects = find_high_grade_objects(stats_collection, grade="Au", min_max=10.0)

# 3. Top 10 objects by Au length-weighted mean
# top_au = get_top_objects_by_grade(stats_collection, grade="Au", metric="lwm", top_n=10)
# for obj in top_au:
#     print(f"  {obj['object_name']}: {obj['lwm']:.4f}")

# 4. Find high-grade objects in a specific workspace
# workspace_high_grade = find_high_grade_objects(
#     stats_collection, 
#     grade="Au", 
#     min_lwm=0.5, 
#     workspace_id=WORKSPACE_ID
# )

Found 1 objects with Au LWM >= 1.0


## 11b. Profile High-Grade Query Performance

Benchmark the high-grade query functions with and without indexes.

In [29]:
# Run the profiling
doc_count = stats_collection.count_documents({})
print(f"Collection has {doc_count} documents\n")

if doc_count > 0:
    # Use first available grade column or default to "Au"
    test_grade = available_grade_columns[0] if 'available_grade_columns' in dir() and available_grade_columns else "Au"
    print(f"Profiling with grade: {test_grade}\n")
    grade_query_results = profile_high_grade_queries(stats_collection, grade=test_grade)
else:
    print("⚠ Insert documents first, then run this cell for profiling")

Collection has 2 documents

Profiling with grade: Au

DROPPING GRADE INDEXES FOR BASELINE...

📊 WITHOUT INDEXES:

   find_high_grade_objects(Au, min_lwm=0.5)
      Mean: 0.9092 ms (±0.1477)
      Docs examined: 2, Index: COLLSCAN

   find_high_grade_objects(Au, min_max=5.0)
      Mean: 0.9378 ms (±0.1358)
      Docs examined: 2, Index: COLLSCAN

   get_top_objects_by_grade(Au, lwm, top_n=10)
      Mean: 0.9223 ms (±0.1178)
      Docs examined: N/A (aggregate), Index: N/A (aggregate)


CREATING GRADE INDEXES...
Created: ['grade_lwm', 'grade_max', 'grade_accumulation', 'workspace_grade_lwm']

📊 WITH INDEXES:

   find_high_grade_objects(Au, min_lwm=0.5)
      Mean: 0.9141 ms (±0.2584)
      Docs examined: 0, Keys: 0, Index: grade_lwm

   find_high_grade_objects(Au, min_max=5.0)
      Mean: 0.8872 ms (±0.1418)
      Docs examined: 1, Keys: 1, Index: grade_max

   get_top_objects_by_grade(Au, lwm, top_n=10)
      Mean: 0.9517 ms (±0.1336)
      Docs examined: N/A, Keys: N/A, Index: N/A (agg